# Fizgig MOSS-Image LoRA Workbench (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/j7708git/colab-workbench/blob/main/lora-training/fizgig_moss.ipynb)

本 Notebook 會在 Colab GPU 上建立隔離環境安裝 `shootthesound/Fizgig`（預設 `v9.7.7`）、批次下載並自動綁定 **MOSS-Image** 訓練／預覽權重至 `user_data/prefs.json`，並支援啟動帶密碼保護的 **Fizgig Web UI**（含 Gizmo 裁圖、Look Consistency 過濾、SAM 3.1 / MediaPipe 人物遮罩、VLM 打標與 LoRA Royale 挑檔）或輸出／執行 CLI 訓練管線。

- **硬體建議**：請使用 **L4** 或 **A100（High-RAM）** 執行階段；不建議使用 T4（無原生 `bf16` Tensor Core 加速，且標準執行階段系統記憶體不足以承載 19.3 GB 的 `qwen3.5_9b_bf16` text cache）。
- **Colab Secrets 設定**：請在左側鑰匙圖示新增 `HF_TOKEN` 並開啟 **Notebook access**，避免共享 IP 觸發 HTTP 429 限流；可選填 `FIZGIG_PASSWORD`（至少 8 字元）作為 Web UI 登入密碼，未填時會於啟動時自動產生隨機密碼。
- **資產隔離**：本公開倉庫不含任何私人資料集或權重路徑；訓練資料與產出 LoRA 請留在本機或個人私有空間，結束後務必執行 `session.stop()` 並釋放 Colab 執行階段。

In [ ]:
#@title 1. 同步 colab-workbench 並載入 HF_TOKEN / 選填 FIZGIG_PASSWORD
import importlib
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/j7708git/colab-workbench.git"
REPO_DIR = "/content/colab-workbench"
FALLBACK_BRANCH = "main"
BRANCH = "main"  #@param {type:"string"}
BRANCH = (BRANCH or FALLBACK_BRANCH).strip() or FALLBACK_BRANCH

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

subprocess.run(
    ["git", "-C", REPO_DIR, "fetch", "origin", f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"],
    check=True,
)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

module_path = os.path.join(REPO_DIR, "lora-training", "fizgig_moss.py")
if not os.path.isfile(module_path) and BRANCH != FALLBACK_BRANCH:
    print(f"[lora-training] 分支 '{BRANCH}' 尚未包含 lora-training/fizgig_moss.py，自動切換至 {FALLBACK_BRANCH} ...")
    BRANCH = FALLBACK_BRANCH
    subprocess.run(
        ["git", "-C", REPO_DIR, "fetch", "origin", f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"],
        check=True,
    )
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

assert os.path.isfile(module_path), f"找不到 {module_path}，請確認分支 {BRANCH} 內容是否完整"

LORA_DIR = os.path.join(REPO_DIR, "lora-training")
desired = [LORA_DIR, REPO_DIR]
sys.path[:] = [*desired, *[p for p in sys.path if p not in desired]]
for _mod in [
    k
    for k in list(sys.modules)
    if k in {"shared", "fizgig_moss"} or k.startswith("shared.")
]:
    del sys.modules[_mod]
importlib.invalidate_caches()

import fizgig_moss
from shared import export_secret_to_env, get_secret_status

fizgig_moss.ensure_fizgig_moss_ready(REPO_DIR)
head_sha = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip()
has_hf_secret, hf_secret_status = get_secret_status("HF_TOKEN")
hf_token = export_secret_to_env("HF_TOKEN", required=False)
_, web_pwd_status = get_secret_status("FIZGIG_PASSWORD")
export_secret_to_env("FIZGIG_PASSWORD", required=False)

print("BRANCH:", BRANCH, f"({head_sha})")
print("fizgig_moss.__file__:", fizgig_moss.__file__)
if hf_token:
    print(f"HF_TOKEN: 已載入（來源：{hf_secret_status}）")
elif "disabled" in hf_secret_status or "access" in hf_secret_status:
    print("HF_TOKEN: 偵測到 Secret 但「Notebook access」未開啟！請至 Colab 左側鑰匙圖示 Secrets 開啟 Notebook access 開關，避免匿名下載觸發 429 限流。")
else:
    print(f"HF_TOKEN: 未設定（{hf_secret_status}）；公開模型可匿名下載，但建議設定 HF_TOKEN 並開啟 Notebook access。")
print(f"FIZGIG_PASSWORD 狀態：{web_pwd_status}（若未設定，啟動 Web UI 時會自動產生一次性隨機密碼）")

In [ ]:
#@title 2. 安裝／更新 Fizgig v9.7.7 與獨立 venv 環境
!bash /content/colab-workbench/lora-training/install.sh

In [ ]:
#@title 3. 下載 MOSS-Image 模型套件並寫入 Fizgig prefs.json
import importlib
import sys

REPO_DIR = "/content/colab-workbench"
LORA_DIR = f"{REPO_DIR}/lora-training"
desired = [LORA_DIR, REPO_DIR]
sys.path[:] = [*desired, *[p for p in sys.path if p not in desired]]
for _mod in [k for k in list(sys.modules) if k in {"shared", "fizgig_moss"} or k.startswith("shared.")]:
    del sys.modules[_mod]
importlib.invalidate_caches()

import fizgig_moss
from shared import get_secret_status

fizgig_moss.ensure_fizgig_moss_ready(REPO_DIR)
PRESET = "standard"  #@param ["standard", "int8_te", "with_fast_preview"]

has_hf_secret, hf_secret_status = get_secret_status("HF_TOKEN")
print(f"HF_TOKEN 狀態：{'已啟用' if has_hf_secret else '匿名模式'}（{hf_secret_status}）")

model_paths = fizgig_moss.download_moss_models(preset=PRESET)
print(f"模型就緒並已寫入 user_data/prefs.json（共 {len(model_paths)} 項）：")
for key, path in model_paths.items():
    print(f" - {key}: {path}")

In [ ]:
#@title 4. 啟動 Fizgig Web UI（或預覽 CLI 訓練指令）
import importlib
import shlex
import sys

REPO_DIR = "/content/colab-workbench"
LORA_DIR = f"{REPO_DIR}/lora-training"
desired = [LORA_DIR, REPO_DIR]
sys.path[:] = [*desired, *[p for p in sys.path if p not in desired]]
for _mod in [k for k in list(sys.modules) if k in {"shared", "fizgig_moss"} or k.startswith("shared.")]:
    del sys.modules[_mod]
importlib.invalidate_caches()

import fizgig_moss

fizgig_moss.ensure_fizgig_moss_ready(REPO_DIR)

MODE = "web_ui"  #@param ["web_ui", "cli_dry_run"]
PORT = 8081  #@param {type:"integer"}
TUNNEL_PROTOCOL = "http2"  #@param ["http2", "auto", "quic"]
RECIPE = "dop_adamw"  #@param ["dop_adamw", "muon_baseline"]
NETWORK_TYPE = "lora"  #@param ["lora", "lokr"]
TRIGGER_WORD = "zwxem"  #@param {type:"string"}
DOP_CLASS = "woman"  #@param {type:"string"}
MAX_TRAIN_EPOCHS = 60  #@param {type:"integer"}
DATASET_DIR = "/content/datasets/sample/train"  #@param {type:"string"}
OUTPUT_NAME = "sample_moss_v1"  #@param {type:"string"}

if MODE == "web_ui":
    session = fizgig_moss.launch_fizgig_web(
        port=PORT,
        tunnel_protocol=TUNNEL_PROTOCOL,
    )
    print("=" * 60)
    print("Fizgig Web UI 已上線（密碼保護）！")
    print("Local URL:     ", session.local_url)
    print("Public URL:    ", session.public_url)
    print("Web Password:  ", session.password)
    print("Fizgig Log:    ", session.log_file)
    print("Tunnel Log:    ", session.tunnel_log_file)
    print("停止服務與隧道請執行：session.stop()")
    print("=" * 60)
else:
    models_dir = fizgig_moss.DEFAULT_FIZGIG_DIR / "user_data" / "models"
    toml_path = fizgig_moss.write_dataset_toml(
        "/dev/shm/preview_dataset.toml",
        image_directory=DATASET_DIR,
        cache_directory=f"/content/Fizgig/user_data/cache/{OUTPUT_NAME}",
    )
    vae_path = models_dir / "flux2-vae_small.safetensors"
    te_path = models_dir / "qwen3.5_9b_bf16.safetensors"
    dit_path = models_dir / "moss_image_base_bf16.safetensors"
    sft_path = models_dir / "moss_image_sft_int8_convrot.safetensors"
    latents_cmd, text_cmd = fizgig_moss.build_moss_cache_commands(
        toml_path,
        vae_path=vae_path,
        text_encoder_path=te_path,
    )
    train_cmd = fizgig_moss.build_moss_train_command(
        fizgig_moss.MossTrainConfig(
            dataset_config=toml_path,
            dit_path=dit_path,
            vae_path=vae_path,
            text_encoder_path=te_path,
            preview_checkpoint=sft_path,
            output_dir=f"/content/Fizgig/output_loras/{OUTPUT_NAME}",
            output_name=OUTPUT_NAME,
            trigger_word=TRIGGER_WORD,
            recipe=RECIPE,
            network_type=NETWORK_TYPE,
            dop_class=DOP_CLASS,
            max_train_epochs=MAX_TRAIN_EPOCHS,
        )
    )
    print("Latents Cache:", shlex.join(latents_cmd))
    print("Text Cache:   ", shlex.join(text_cmd))
    print("Train Cmd:    ", shlex.join(train_cmd))